# Review links and explore the network

Shows IRS grant and related-org networks, LDA lobbying data, and cross-source views built from accepted name matches.


## Setup


In [ ]:
from pathlib import Path
import sqlite3
import sys
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
DB_PATH = ROOT / 'data' / 'irs990_full.db'
conn = sqlite3.connect(DB_PATH)

def q(sql, **kw):
    return pd.read_sql_query(sql, conn, **kw)

# Quick row counts across all tables
for tbl in ['irs990_filings','committees','fec_disbursements','lda_filings',
            'lda_lobbying_activities','bills','entity_observations',
            'entity_match_candidates','entity_match_decisions']:
    n = conn.execute(f'SELECT COUNT(*) FROM {tbl}').fetchone()[0]
    print(f'{tbl:35} {n:>12,}')


## Review name-match candidates

Each row is a candidate pair. Score orders the queue; it is not proof of shared identity. Accept or reject below.


In [ ]:
# Distinct org-name pairs ranked by number of matching filing records
# (one org can appear in many filings; GROUP BY collapses to unique org pairs)
n_total = conn.execute('SELECT COUNT(*) FROM entity_match_candidates WHERE is_current=1').fetchone()[0]
n_pairs = conn.execute('''
    SELECT COUNT(*) FROM (
        SELECT lo.normalized_name, ro.normalized_name
        FROM entity_match_candidates c
        JOIN entity_observations lo ON lo.observation_id = c.left_observation_id
        JOIN entity_observations ro ON ro.observation_id = c.right_observation_id
        WHERE c.is_current=1 AND lo.source_system='IRS990'
        GROUP BY lo.normalized_name, ro.normalized_name
    )
''').fetchone()[0]
print(f'Total match candidates (filing-level): {n_total:,}')
print(f'Distinct org pairs:                    {n_pairs:,}')
print()
candidates = q('''
    SELECT MIN(c.candidate_id)         AS candidate_id,
           lo.observed_name            AS irs_name,
           ro.observed_name            AS ext_name,
           ro.source_system            AS ext_source,
           c.matcher_name,
           c.score,
           COUNT(*)                    AS filing_pairs
    FROM entity_match_candidates AS c
    JOIN entity_observations AS lo ON lo.observation_id = c.left_observation_id
    JOIN entity_observations AS ro ON ro.observation_id = c.right_observation_id
    WHERE c.is_current = 1
      AND lo.source_system = \'IRS990\'
    GROUP BY lo.normalized_name, ro.normalized_name, ro.source_system, c.matcher_name
    ORDER BY filing_pairs DESC, c.score DESC
    LIMIT 50
''')
candidates


### How cross-source connections are made

IRS, FEC, and LDA share no common ID. The pipeline connects them two ways:

**1. Name matching (needs human sign-off)**

Every org name from each source is normalized (lowercase, punctuation removed, noise words like 'Inc' and 'LLC' stripped) and stored in `entity_observations`. Pairs with identical normalized names are queued in `entity_match_candidates` with score 1.0. Fuzzy matching is available separately but is slow at this scale.

Nothing reaches the analysis views until a decision is written to `entity_match_decisions`. The `approved_external_entity_links` view assembles `(IRS EIN, source system, external record ID)` triples from accepted decisions only.

**2. Bill references from lobbying text (no review needed)**

LDA activity descriptions often name bills directly, e.g. `HR 2670` or `S.1001`. A regex scans every description and writes matches to `lobbying_bill_links`. No name matching involved.

**What each view requires**

| View | How it joins | Needs accepted matches? |
|---|---|---|
| `grant_network_edges` | IRS Sch. I grantee EINs | No |
| `related_organization_edges` | IRS Sch. R related EINs | No |
| `lobbying_bill_facts` | LDA filing -> bill regex | No |
| `organization_policy_links` | IRS EIN -> accepted LDA filing -> bill | Yes |
| `organization_fec_disbursements` | IRS EIN -> accepted FEC committee -> disbursements | Yes |

**Auto-accepting identical name matches**

The cell below accepts all IRS<->LDA pairs where the org name is the same text, differing only in capitalization (IRS uses title-case; LDA submissions are all-caps). FEC is excluded because generic words like 'Freedom' or 'Fund' produce many false matches between nonprofits and unrelated PACs. Those need the manual review queue above.


In [ ]:
# Auto-accept IRS<->LDA candidates where observed names are identical (case-insensitive).
# Idempotent: INSERT OR IGNORE means re-running creates no duplicates.
# FEC candidates excluded -- generic name tokens cause false positives with PACs.

already = conn.execute(
    "SELECT COUNT(*) FROM entity_match_decisions WHERE reviewer='auto-exact-text'"
).fetchone()[0]

if already > 0:
    print(f'Already applied: {already:,} auto-exact-text decisions on record.')
else:
    rows = conn.execute('''
        SELECT c.candidate_id
        FROM entity_match_candidates AS c
        JOIN entity_observations AS lo ON lo.observation_id = c.left_observation_id
        JOIN entity_observations AS ro ON ro.observation_id = c.right_observation_id
        WHERE c.is_current = 1
          AND lo.source_system = 'IRS990'
          AND ro.source_system = 'LDA'
          AND c.score = 1.0
          AND UPPER(lo.observed_name) = UPPER(ro.observed_name)
    ''').fetchall()
    conn.executemany(
        "INSERT OR IGNORE INTO entity_match_decisions "
        "(candidate_id, decision, reviewer, rationale) "
        "VALUES (?, 'accepted', 'auto-exact-text', "
        "'Identical observed_name in IRS990 and LDA (case-insensitive); auto-accepted.')",
        rows
    )
    conn.commit()
    print(f'Accepted {len(rows):,} IRS<->LDA exact-text match candidates.')

n_decisions = conn.execute("SELECT COUNT(*) FROM entity_match_decisions WHERE decision='accepted'").fetchone()[0]
n_links = conn.execute('SELECT COUNT(*) FROM organization_policy_links').fetchone()[0]
print(f'Total accepted decisions:       {n_decisions:,}')
print(f'organization_policy_links rows: {n_links:,}')


## IRS grant network

Grant flows from Schedule I, joined on grantee EIN. No name matching needed.


In [ ]:
grant_edges = q('''
    SELECT source_ein, target_ein, amount, supporting_rows
    FROM grant_network_edges
    ORDER BY amount DESC
    LIMIT 100
''')
print(f'Total grant edges: {conn.execute("SELECT COUNT(*) FROM grant_network_edges").fetchone()[0]:,}')
grant_edges.head(20)


## IRS related-organization network

Related-org edges from Schedule R, joined on EIN.


In [ ]:
related_edges = q('''
    SELECT source_ein, target_ein, supporting_rows
    FROM related_organization_edges
    ORDER BY supporting_rows DESC
    LIMIT 100
''')
print(f'Total related-org edges: {conn.execute("SELECT COUNT(*) FROM related_organization_edges").fetchone()[0]:,}')
related_edges.head(20)


## LDA lobbying filings

Top spending clients, issue code breakdown, and bill references from activity text.


In [ ]:
# Top spenders by total expenses
top_spenders = q('''
    SELECT client_name, SUM(CAST(expenses AS REAL)) AS total_expenses,
           COUNT(*) AS filings
    FROM lda_filings
    WHERE expenses IS NOT NULL
    GROUP BY client_name
    ORDER BY total_expenses DESC
    LIMIT 20
''')
top_spenders


In [ ]:
# Issue code distribution
issue_codes = q('''
    SELECT general_issue_code, COUNT(*) AS activity_count
    FROM lda_lobbying_activities
    WHERE general_issue_code IS NOT NULL
    GROUP BY general_issue_code
    ORDER BY activity_count DESC
    LIMIT 30
''')
issue_codes


## Cross-source views

Populated once matches are accepted above. Run `refresh_analysis_layers(DB_PATH)` in notebook 01 after adding new decisions.


In [ ]:
policy_links = q('''
    SELECT ein, client_name, bill_id, title, filing_year,
           reported_lobbying_amount
    FROM organization_policy_links
    ORDER BY reported_lobbying_amount DESC
    LIMIT 100
''')
fec_spending = q('SELECT * FROM organization_fec_disbursements LIMIT 100')
print(f'Policy links (IRS org -> bills via LDA):   {conn.execute("SELECT COUNT(*) FROM organization_policy_links").fetchone()[0]:,} rows')
print(f'FEC spending (IRS org -> disbursements):   {conn.execute("SELECT COUNT(*) FROM organization_fec_disbursements").fetchone()[0]:,} rows')
print()
print('Top lobbying orgs by reported spend -> bill:')
policy_links.head(20)


## Congress bills

Bills are in stub mode (title and bill type only). Call `congress.collect_bill_detail(bill_id)` for full detail on a specific bill.


In [ ]:
# Bills are in stub mode (introduced_date, policy_area not populated from Congress API)
# Browse by type or search by bill number using lobbying_bill_facts for cross-reference
print(f'Total bills: {conn.execute("SELECT COUNT(*) FROM bills").fetchone()[0]:,}')
print(f'With policy_area: {conn.execute("SELECT COUNT(*) FROM bills WHERE policy_area IS NOT NULL").fetchone()[0]:,}')
print()
bills_by_type = q('''
    SELECT bill_type, COUNT(*) AS bills
    FROM bills
    GROUP BY bill_type
    ORDER BY bills DESC
''')
print('Bills by type:')
display(bills_by_type)

# Sample of actual bills
bills_sample = q('''
    SELECT bill_type, bill_number, title, introduced_date, latest_action
    FROM bills
    ORDER BY bill_number
    LIMIT 20
''')
bills_sample
